In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# ГП СП


In [8]:
territory_gdf = gpd.read_file('data/level_4_territories.geojson').to_crs(32636)
territory_gdf = territory_gdf[['territory_id', 'name', 'geometry']]
territory_gdf

,territory_id,name,geometry
0,3,Самойловское сельское поселение,"POLYGON ((580049.515 6617067.72, 580225.344 66..."
1,4,Большедворское сельское поселение,"POLYGON ((573514.444 6620389.851, 573579.919 6..."
2,5,Пикалевское городское поселение,"POLYGON ((564707.328 6594903.594, 564341.814 6..."
3,6,Борское сельское поселение,"POLYGON ((560629.509 6557332.762, 560642.857 6..."
4,7,Бокситогорское городское поселение,"POLYGON ((560659.317 6585430.874, 559919.213 6..."
...,...,...,...
184,204,Красноборское городское поселение,"POLYGON ((363803.737 6614904.469, 363821.597 6..."
185,205,Трубникоборское сельское поселение,"POLYGON ((392924.072 6551507.217, 393174.409 6..."
186,206,Лисинское сельское поселение,"POLYGON ((363959.223 6596385.107, 364013.066 6..."
187,1267,город Сосновый бор,"POLYGON ((283435.008 6647206.979, 283361.394 6..."


In [6]:
import requests
import geopandas as gpd
import pandas as pd

# Загрузка вашего GeoDataFrame
gdf = territory_gdf

# API-адрес
URBAN_API = "https://urban-api.idu.kanootoko.org/api/v1"

# Словарь для хранения всех значений индикаторов
all_indicators = {}

for territory_id in gdf["territory_id"]:
    try:
        # Формируем URL
        url = f"{URBAN_API}/territory/{territory_id}/indicator_values"
        
        # Выполняем GET-запрос
        response = requests.get(url)
        
        # Проверяем статус ответа
        if response.status_code != 200:
            print(f"Ошибка при запросе {territory_id}: {response.status_code}")
            continue
        
        # Обрабатываем JSON-ответ
        res_json = response.json()

        # Собираем индикаторы в словарь
        indicators = {item["indicator"]["name_full"]: item["value"] for item in res_json}
        all_indicators[territory_id] = indicators

    except Exception as e:
        print(f"Ошибка при обработке {territory_id}: {e}")
        continue

# Преобразуем в DataFrame
indicators_df = pd.DataFrame.from_dict(all_indicators, orient="index").reset_index()
indicators_df.rename(columns={"index": "territory_id"}, inplace=True)

# Объединяем с исходным GeoDataFrame
gdf = gdf.merge(indicators_df, on="territory_id", how="left")

gdf


,territory_id,name,geometry,Численность населения,Степень урбанизации территории,Средняя доступность до федеральных транспортных магистралей,Средняя доступность до центра региона,Связанность,Связанность населенных пунктов,Плотность улично-дорожной сети,...,Средняя доступность до центра федерального округа,Социальное обеспечение (базовое),Социальное обеспечение (базовое+),Социальное обеспечение (комфорт),Количество объектов инженерной инфраструктуры,"Количество электростанций (ТЭС, АЭС и пр.)",Количество больших водозаборов,Количество очистительных сооружений,Количество крупных водохранилищ,Количество газораспределительных станций
0,3,Самойловское сельское поселение,"POLYGON ((580049.515 6617067.72, 580225.344 66...",2154.0,23.68,6.088,307.405291,4.004826,239.5200,0.113,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,Большедворское сельское поселение,"POLYGON ((573514.444 6620389.851, 573579.919 6...",1698.0,49.15,18.029,278.750032,3.619422,216.3970,0.142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,Пикалевское городское поселение,"POLYGON ((564707.328 6594903.594, 564341.814 6...",20169.0,45.76,0.084,290.259052,3.738074,223.5170,1.168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,Борское сельское поселение,"POLYGON ((560629.509 6557332.762, 560642.857 6...",3393.0,51.29,26.426,293.624908,3.826036,228.9690,0.106,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,Бокситогорское городское поселение,"POLYGON ((560659.317 6585430.874, 559919.213 6...",15960.0,33.44,18.846,280.784975,3.610566,215.8650,0.281,...,252.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,204,Красноборское городское поселение,"POLYGON ((363803.737 6614904.469, 363821.597 6...",4507.0,33.85,3.379,59.148080,2.048569,122.8555,0.954,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
185,205,Трубникоборское сельское поселение,"POLYGON ((392924.072 6551507.217, 393174.409 6...",1620.0,65.34,8.596,102.933623,2.566700,153.9260,0.198,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,206,Лисинское сельское поселение,"POLYGON ((363959.223 6596385.107, 364013.066 6...",1919.0,78.50,32.716,59.644562,2.337061,140.3240,0.101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,1267,город Сосновый бор,"POLYGON ((283435.008 6647206.979, 283361.394 6...",63462.0,66.50,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,6.0,0.0,0.0,5.0,1.0,0.0


In [9]:
gdf.to_parquet('data/territories_indicators.parquet')

In [23]:
columns_with_nan = gdf.columns.tolist()
columns_with_nan

['territory_id',
 'name',
 'geometry',
 'Численность населения',
 'Степень урбанизации территории',
 'Средняя доступность до федеральных транспортных магистралей',
 'Средняя доступность до центра региона',
 'Связанность',
 'Связанность населенных пунктов',
 'Плотность улично-дорожной сети',
 'Протяженность дорог федерального значения',
 'Протяженность дорог регионального значения',
 'Протяженность дорог местного значения',
 'Количество маршрутов общественного транспорта',
 'Количество остановок общественного транспорта',
 'Количество автозаправочных станций',
 'Средняя доступность  автозаправочных станций',
 'Общая протяженность железнодорожных путей',
 'Количество остановок железнодорожного транспорта',
 'Средняя доступность остановок железнодорожного транспорта',
 'Количество международных аэропортов',
 'Количество аэропортов местного значения',
 'Средняя доступность международных аэропортов',
 'Средняя доступность аэропортов местного значения',
 'Количество портов',
 'Обеспеченность

In [ ]:
import requests
import geopandas as gpd

URBAN_API = "https://urban-api.idu.kanootoko.org/api/v1"
params = {
    "territory_id" : 3,
}

# Формируем URL для конкретной территории
territory_id = 1  # Пример ID территории
url = f"{URBAN_API}/territory/{territory_id}/indicator_values"

# Выполняем GET-запрос
response = requests.get(url, params=params)

res_json = response.json()
res_json

# Кварталы

In [2]:
territories_gdf = gpd.read_parquet('data/merged_data.parquet').to_crs(32636)
territories_gdf

,geometry,x,y,area,length,corners_count,outer_radius,inner_radius,aspect_ratio,residential,business,recreation,industrial,transport,special,agriculture,id
0,"POLYGON ((572511.69 6596566.888, 572396.077 65...",571890.770590,6.596896e+06,1.378104e+06,5038.253473,93,1027.674666,387.918839,2.088696,0.387353,0.000000,0.029147,0.092012,0.0,0.000000,0.0,0
1,"POLYGON ((571198.152 6596469.512, 571174.429 6...",571053.592580,6.596457e+06,1.210807e+03,469.652318,27,131.163697,4.881751,6.135552,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1
2,"POLYGON ((572867.931 6596649.534, 572806.268 6...",572753.824071,6.596626e+06,1.007658e+03,440.310836,21,128.091315,14.602166,6.145173,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,2
3,"POLYGON ((583241.896 6595452.983, 583366.911 6...",583400.357747,6.595371e+06,4.612804e+04,939.249125,19,182.364505,67.321517,1.997666,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,3
4,"POLYGON ((582403.742 6595460.48, 582445.687 65...",582416.423405,6.595452e+06,2.886984e+02,104.471470,10,27.715236,6.358547,2.087689,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30093,"POLYGON ((282882.262 6646551.855, 282870.234 6...",618450.818860,6.642078e+06,3.803840e+02,85.246434,9,21.834857,7.176879,2.149529,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,30093
30094,"POLYGON ((283260.086 6646763.048, 283264.203 6...",618764.184666,6.642262e+06,2.113179e+04,855.808176,20,209.111662,46.179469,3.654009,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,30094
30095,"POLYGON ((283218.856 6646746.21, 283023.775 66...",618675.714935,6.642287e+06,1.380902e+04,542.502261,6,111.965346,31.203913,2.781157,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,30095
30096,"POLYGON ((282918.205 6646642.759, 282890.041 6...",618529.762899,6.642217e+06,7.680389e+03,383.127822,9,82.713797,29.215515,2.576965,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,30096


In [4]:
# road_gdf = gpd.read_file('data/road.geojson').to_crs(32636)

In [3]:
road_buffer_gdf = gpd.read_parquet('data/road_buffer.parquet').to_crs(32636)
road_buffer_gdf

,geometry
0,"MULTIPOLYGON (((227443.293 6540592.627, 227443..."


In [ ]:
# import geopandas as gpd
# from shapely.ops import unary_union


# # Применяем overlay (разница между территориями и буфером)
# split_gdf = gpd.overlay(territories_gdf, road_buffer_gdf, how='difference')
# split_gdf = split_gdf.explode(index_parts=False).reset_index(drop=True)
# split_gdf['geometry'] = split_gdf.buffer(5).buffer(-5)

# split_gdf.to_parquet('split_gdf.parquet')

  

In [10]:
split_gdf = gpd.read_parquet('data/split_gdf.parquet').to_crs(32636)
split_gdf = split_gdf[['geometry']]
split_gdf

,geometry
0,"POLYGON ((570883.045 6596529.946, 570883.081 6..."
1,"POLYGON ((570987.035 6596464.499, 570987.085 6..."
2,"POLYGON ((572676.73 6596606.982, 572676.701 65..."
3,"POLYGON ((583242.895 6595451.954, 583243.013 6..."
4,"POLYGON ((582405.498 6595459.455, 582405.614 6..."
...,...
34645,"POLYGON ((279466.739 6647657.982, 279466.837 6..."
34646,"POLYGON ((279467.148 6647660.126, 279467.165 6..."
34647,"POLYGON ((279542.296 6648001.072, 279542.336 6..."
34648,"POLYGON ((279625.847 6648207.164, 279625.886 6..."


In [12]:
split_gdf.to_parquet('split_gdf.parquet')


In [11]:
split_gdf = split_gdf.reset_index().rename(columns={'index': 'id'})
split_gdf

,id,geometry
0,0,"POLYGON ((570883.045 6596529.946, 570883.081 6..."
1,1,"POLYGON ((570987.035 6596464.499, 570987.085 6..."
2,2,"POLYGON ((572676.73 6596606.982, 572676.701 65..."
3,3,"POLYGON ((583242.895 6595451.954, 583243.013 6..."
4,4,"POLYGON ((582405.498 6595459.455, 582405.614 6..."
...,...,...
34645,34645,"POLYGON ((279466.739 6647657.982, 279466.837 6..."
34646,34646,"POLYGON ((279467.148 6647660.126, 279467.165 6..."
34647,34647,"POLYGON ((279542.296 6648001.072, 279542.336 6..."
34648,34648,"POLYGON ((279625.847 6648207.164, 279625.886 6..."


# ML - реальные данные

In [3]:
gdf = gpd.read_parquet('data/split_gdf.parquet').to_crs(32636)
gdf

,id,geometry
0,0,"POLYGON ((570883.045 6596529.946, 570883.081 6..."
1,1,"POLYGON ((570987.035 6596464.499, 570987.085 6..."
2,2,"POLYGON ((572676.73 6596606.982, 572676.701 65..."
3,3,"POLYGON ((583242.895 6595451.954, 583243.013 6..."
4,4,"POLYGON ((582405.498 6595459.455, 582405.614 6..."
...,...,...
34645,34645,"POLYGON ((279466.739 6647657.982, 279466.837 6..."
34646,34646,"POLYGON ((279467.148 6647660.126, 279467.165 6..."
34647,34647,"POLYGON ((279542.296 6648001.072, 279542.336 6..."
34648,34648,"POLYGON ((279625.847 6648207.164, 279625.886 6..."


In [4]:
gdf = gpd.read_parquet('data/territories_indicators.parquet')
gdf.head()

FileNotFoundError: [Errno 2] Failed to open local file 'data/territories_indicators.parquet'. Detail: [errno 2] No such file or directory

In [32]:
filtered_gdf = gdf[gdf['Социальное обеспечение (базовое+)'].notna()]
filtered_gdf

,territory_id,name,geometry,Численность населения,Степень урбанизации территории,Средняя доступность до федеральных транспортных магистралей,Средняя доступность до центра региона,Связанность,Связанность населенных пунктов,Плотность улично-дорожной сети,...,Средняя доступность до центра федерального округа,Социальное обеспечение (базовое),Социальное обеспечение (базовое+),Социальное обеспечение (комфорт),Количество объектов инженерной инфраструктуры,"Количество электростанций (ТЭС, АЭС и пр.)",Количество больших водозаборов,Количество очистительных сооружений,Количество крупных водохранилищ,Количество газораспределительных станций
102,113,Шлиссельбургское городское поселение,"POLYGON ((390395.147 6648147.059, 401096.266 6...",13918.0,33.72,6.924,92.992334,2.199687,132.051,0.538,...,NaN,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
columns_with_nan = gdf.columns.tolist()
columns_with_nan

['territory_id',
 'name',
 'geometry',
 'Численность населения',
 'Степень урбанизации территории',
 'Средняя доступность до федеральных транспортных магистралей',
 'Средняя доступность до центра региона',
 'Связанность',
 'Связанность населенных пунктов',
 'Плотность улично-дорожной сети',
 'Протяженность дорог федерального значения',
 'Протяженность дорог регионального значения',
 'Протяженность дорог местного значения',
 'Количество маршрутов общественного транспорта',
 'Количество остановок общественного транспорта',
 'Количество автозаправочных станций',
 'Средняя доступность  автозаправочных станций',
 'Общая протяженность железнодорожных путей',
 'Количество остановок железнодорожного транспорта',
 'Средняя доступность остановок железнодорожного транспорта',
 'Количество международных аэропортов',
 'Количество аэропортов местного значения',
 'Средняя доступность международных аэропортов',
 'Средняя доступность аэропортов местного значения',
 'Количество портов',
 'Обеспеченность

In [46]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Инициализация LabelEncoder
le = LabelEncoder()
land_use_types = ['residential', 'recreation', 'special', 'industrial', 
                 'agriculture', 'transport', 'business', 'basic',
                 'residential_individual', 'residential_lowrise', 
                 'residential_midrise', 'residential_multistorey', 'mixed_use']
le.fit(land_use_types)

# Эффекты типов землепользования
land_use_effect = {
    'residential': 0.7, 'recreation': 0.4, 'special': 0.3, 'industrial': 1.2,
    'agriculture': 0.5, 'transport': 0.8, 'business': 1.6, 'basic': 0.2,
    'residential_individual': 0.9, 'residential_lowrise': 1.0,
    'residential_midrise': 1.1, 'residential_multistorey': 1.3, 'mixed_use': 1.5
}

import numpy as np
from scipy.spatial import cKDTree

# Предполагаем, что ваш GDF называется gdf
numeric_columns = gdf.select_dtypes(include=[np.number]).columns

# Функция для заполнения NaN на основе ближайших соседей
def fill_nan_with_neighbors(gdf, column):
    # Извлекаем координаты центроидов геометрий
    coords = np.array([(geom.centroid.x, geom.centroid.y) for geom in gdf.geometry])
    
    # Создаем KDTree для быстрого поиска ближайших соседей
    tree = cKDTree(coords)
    
    # Копия значений столбца
    values = gdf[column].copy()
    
    # Индексы строк с NaN
    nan_indices = gdf.index[gdf[column].isna()]
    
    for idx in nan_indices:
        # Находим ближайших соседей (например, 5 ближайших)
        distances, neighbor_indices = tree.query(coords[idx], k=5)
        
        # Берем значения у соседей
        neighbor_values = gdf[column].iloc[neighbor_indices]
        
        # Если есть не-NaN значения у соседей, берем среднее
        valid_values = neighbor_values.dropna()
        if not valid_values.empty:
            values[idx] = valid_values.mean()
        else:
            # Если у всех соседей NaN, ставим 0
            values[idx] = 0
            
    return values

# Применяем функцию к каждому числовому столбцу
for col in numeric_columns:
    gdf[col] = fill_nan_with_neighbors(gdf, col)
# Создаем словарь всех признаков
features = {
    'Population': gdf['Численность населения'],
    'Urbanization': gdf['Степень урбанизации территории'],
    'HighwayAccessibility': gdf['Средняя доступность до федеральных транспортных магистралей'],
    'RegionCenterAccessibility': gdf['Средняя доступность до центра региона'],
    'Connectivity': gdf['Связанность'],
    'SettlementConnectivity': gdf['Связанность населенных пунктов'],
    'RoadDensity': gdf['Плотность улично-дорожной сети'],
    'FederalRoads': gdf['Протяженность дорог федерального значения'],
    'RegionalRoads': gdf['Протяженность дорог регионального значения'],
    'LocalRoads': gdf['Протяженность дорог местного значения'],
    'PublicTransportRoutes': gdf['Количество маршрутов общественного транспорта'],
    'PublicTransportStops': gdf['Количество остановок общественного транспорта'],
    'GasStations': gdf['Количество автозаправочных станций'],
    'GasStationAccessibility': gdf['Средняя доступность  автозаправочных станций'],
    'RailwayLength': gdf['Общая протяженность железнодорожных путей'],
    'RailwayStops': gdf['Количество остановок железнодорожного транспорта'],
    'RailwayAccessibility': gdf['Средняя доступность остановок железнодорожного транспорта'],
    'IntAirports': gdf['Количество международных аэропортов'],
    'LocalAirports': gdf['Количество аэропортов местного значения'],
    'IntAirportAccessibility': gdf['Средняя доступность международных аэропортов'],
    'LocalAirportAccessibility': gdf['Средняя доступность аэропортов местного значения'],
    'Ports': gdf['Количество портов'],
    'Kindergartens': gdf['Обеспеченность детскими садами'],
    'Schools': gdf['Обеспеченность школами'],
    'ChildrenCenters': gdf['Обеспеченность центрами детского творчества'],
    'ArtSchools': gdf['Обеспеченность школами искусств'],
    'MedicalPoints': gdf['Обеспеченность ФАП / амбулаториями'],
    'Pharmacies': gdf['Обеспеченность аптеками'],
    'Clinics': gdf['Обеспеченность поликлиниками / центрами семейной медицины'],
    'LocalHospitals': gdf['Обеспеченность участковыми больницами'],
    'CityHospitals': gdf['Обеспеченность городскими больницами (д/в)'],
    'CommercialClinics': gdf['Обеспеченность коммерческими клиниками'],
    'GroceryStores': gdf['Обеспеченность продуктовыми магазинами'],
    'HardwareStores': gdf['Обеспеченность хозяйственными магазинами'],
    'ShoppingCenters': gdf['Обеспеченность ТЦ / супермаркетами'],
    'PetStores': gdf['Обеспеченность зоомагазинами'],
    'Hypermarkets': gdf['Обеспеченность ТРК / гипермаркетами'],
    'SpecialtyStores': gdf['Обеспеченность профильными магазинами'],
    'Cafes': gdf['Обеспеченность кафе / кофейнями'],
    'Restaurants': gdf['Обеспеченность барами / ресторанами'],
    'FoodCourts': gdf['Обеспеченность фудкортами'],
    'CommunityHalls': gdf['Обеспеченность универсальными залами'],
    'CultureCenters': gdf['Обеспеченность комьюнити-центрами / домами культуры'],
    'Libraries': gdf['Обеспеченность медиатеками / библиотеками'],
    'ReligiousSites': gdf['Обеспеченность культовыми объектами'],
    'Playgrounds': gdf['Обеспеченность детскими площадками'],
    'Parks': gdf['Обеспеченность скверами / бульварами / лесопарками'],
    'PublicSpaces': gdf['Обеспеченность общественными пространства'],
    'LargeParks': gdf['Обеспеченность парками'],
    'AmusementParks': gdf['Обеспеченность парками развлечений'],
    'DogParks': gdf['Обеспеченность площадками для выгула собак'],
    'WorkoutAreas': gdf['Обеспеченность воркаутами / школьными спортзалами'],
    'FitnessCenters': gdf['Обеспеченность спортзалами ОП / фитнес-центрами'],
    'SkateParks': gdf['Обеспеченность скейтпарками / воркаутами для подростков'],
    'SportsComplexes': gdf['Обеспеченность ФОК / бассейнами'],
    'DeliveryPoints': gdf['Обеспеченность пунктами доставки / почтовыми отделениями'],
    'Salons': gdf['Обеспеченность парикмахерскими / салонами красоты'],
    'HouseholdServices': gdf['Обеспеченность бытовые услугами'],
    'Banks': gdf['Обеспеченность отделениями банков'],
    'TransportStops': gdf['Обеспеченность остановками ОТ'],
    'Parking': gdf['Обеспеченность парковками'],
    'FuelStations': gdf['Обеспеченность автозаправками'],
    'TrainStations': gdf['Обеспеченность ЖД станциями'],
    'CarServices': gdf['Обеспеченность автосалонами / автосервисами'],
    'PolicePoints': gdf['Обеспеченность участковыми пунктами полиции'],
    'PoliceStations': gdf['Обеспеченность опорными пунктами полиции'],
    'FireStations': gdf['Обеспеченность пожарными депо'],
    'PublicTransportConnectivity': gdf['Связанность общественным транспортом'],
    'FederalCenterAccessibility': gdf['Средняя доступность до федерального центра'],
    'DistrictCenterAccessibility': gdf['Средняя доступность до центра федерального округа'],
    'SocialBasic': gdf['Социальное обеспечение (базовое)'],
    'SocialPlus': gdf['Социальное обеспечение (базовое+)'],
    'SocialComfort': gdf['Социальное обеспечение (комфорт)'],
    'InfrastructureObjects': gdf['Количество объектов инженерной инфраструктуры'],
    'PowerPlants': gdf['Количество электростанций (ТЭС, АЭС и пр.)'],
    'WaterIntakes': gdf['Количество больших водозаборов'],
    'TreatmentFacilities': gdf['Количество очистительных сооружений'],
    'Reservoirs': gdf['Количество крупных водохранилищ'],
    'GasStationsDist': gdf['Количество газораспределительных станций']
}

# Создаем DataFrame для обучения
training_data = []
for idx, row in gdf.iterrows():
    for land_type in land_use_types:
        sample = {'LandUseType': land_type}
        for feature_name, feature_series in features.items():
            sample[feature_name] = feature_series.iloc[idx]
        training_data.append(sample)

df = pd.DataFrame(training_data)

# Обработка данных
for col in df.columns:
    if col != 'LandUseType':
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
        df[col] = df[col].clip(lower=0, upper=1e6)

# Расширенная формула инвестиционной привлекательности
# Расширенная формула инвестиционной привлекательности
df['InvestmentReturn'] = (
    # Демография и урбанизация (15%)
    0.03 * df['Population'].clip(0, 1e5) / 1000 +                     # Численность населения
    0.03 * df['Urbanization'].clip(0, 100) / 100 +                    # Степень урбанизации
    0.03 * np.log1p(df['RoadDensity'].clip(0, 100)) +                # Плотность дорожной сети
    0.03 * df['SettlementConnectivity'].clip(0, 100) / 100 +          # Связанность населённых пунктов
    0.03 * df['PublicSpaces'].clip(0, 100) / 10 +                     # Общественные пространства
    
    # Транспортная инфраструктура (25%)
    0.03 * df['FederalRoads'].clip(0, 10000) / 1000 +                 # Федеральные дороги
    0.03 * df['RegionalRoads'].clip(0, 10000) / 1000 +                # Региональные дороги
    0.03 * df['LocalRoads'].clip(0, 10000) / 1000 +                   # Местные дороги
    0.02 * df['PublicTransportRoutes'].clip(0, 1000) / 100 +          # Маршруты общественного транспорта
    0.02 * df['PublicTransportStops'].clip(0, 1000) / 100 +           # Остановки общественного транспорта
    0.02 * df['RailwayLength'].clip(0, 10000) / 1000 +                # Железнодорожные пути
    0.02 * df['RailwayStops'].clip(0, 1000) / 100 +                   # ЖД остановки
    0.02 * (1 - df['HighwayAccessibility'].clip(0, 1000) / 1000) +    # Доступность магистралей
    0.02 * (1 - df['RailwayAccessibility'].clip(0, 1000) / 1000) +    # Доступность ЖД остановок
    0.02 * df['IntAirports'].clip(0, 10) +                            # Международные аэропорты
    0.02 * df['LocalAirports'].clip(0, 10) +                          # Местные аэропорты
    0.02 * df['Ports'].clip(0, 10) +                                  # Порты
    
    # Социальная инфраструктура (25%)
    0.02 * df['Kindergartens'].clip(0, 100) / 10 +                    # Детские сады
    0.02 * df['Schools'].clip(0, 100) / 10 +                          # Школы
    0.02 * df['ChildrenCenters'].clip(0, 100) / 10 +                  # Центры детского творчества
    0.02 * df['ArtSchools'].clip(0, 100) / 10 +                       # Школы искусств
    0.02 * df['MedicalPoints'].clip(0, 100) / 10 +                    # ФАП/амбулатории
    0.02 * df['Pharmacies'].clip(0, 100) / 10 +                       # Аптеки
    0.02 * df['Clinics'].clip(0, 100) / 10 +                          # Поликлиники
    0.02 * df['LocalHospitals'].clip(0, 100) / 10 +                   # Участковые больницы
    0.02 * df['CityHospitals'].clip(0, 100) / 10 +                    # Городские больницы
    0.01 * df['Playgrounds'].clip(0, 100) / 10 +                      # Детские площадки
    0.01 * df['Parks'].clip(0, 100) / 10 +                            # Парки
    0.01 * df['LargeParks'].clip(0, 100) / 10 +                       # Большие парки
    0.01 * df['AmusementParks'].clip(0, 100) / 10 +                   # Парки развлечений
    0.01 * df['SocialComfort'].clip(0, 1000) / 100 +                  # Соцобеспечение комфорт
    
    # Экономические факторы (20%)
    0.02 * df['GroceryStores'].clip(0, 100) / 10 +                    # Продуктовые магазины
    0.02 * df['HardwareStores'].clip(0, 100) / 10 +                   # Хозяйственные магазины
    0.02 * df['ShoppingCenters'].clip(0, 100) / 10 +                  # ТЦ/супермаркеты
    0.02 * df['Hypermarkets'].clip(0, 100) / 10 +                     # Гипермаркеты
    0.02 * df['SpecialtyStores'].clip(0, 100) / 10 +                  # Профильные магазины
    0.02 * df['Cafes'].clip(0, 100) / 10 +                            # Кафе
    0.02 * df['Restaurants'].clip(0, 100) / 10 +                      # Рестораны
    0.02 * df['CommercialClinics'].clip(0, 100) / 10 +                # Коммерческие клиники
    0.02 * df['Banks'].clip(0, 100) / 10 +                            # Банки
    0.02 * df['GasStations'].clip(0, 100) / 10 +                      # АЗС
    
    # Связанность и доступность (15%)
    0.03 * df['Connectivity'].clip(0, 100) / 100 +                    # Общая связанность
    0.02 * (1 - df['RegionCenterAccessibility'].clip(0, 1000) / 1000) + # Доступность центра региона
    0.02 * (1 - df['FederalCenterAccessibility'].clip(0, 1000) / 1000) + # Доступность федерального центра
    0.02 * (1 - df['DistrictCenterAccessibility'].clip(0, 1000) / 1000) + # Доступность центра округа
    0.02 * df['PublicTransportConnectivity'].clip(0, 100) / 100 +      # Связанность транспортом
    0.02 * df['InfrastructureObjects'].clip(0, 1000) / 100 +           # Инфраструктурные объекты
    0.02 * df['PowerPlants'].clip(0, 100) / 10 +                      # Электростанции
    
    # Эффект типа землепользования
    df['LandUseType'].map(land_use_effect) +
    # Случайный шум
    np.random.normal(0, 0.5, len(df))
)

df

,LandUseType,Population,Urbanization,HighwayAccessibility,RegionCenterAccessibility,Connectivity,SettlementConnectivity,RoadDensity,FederalRoads,RegionalRoads,...,SocialBasic,SocialPlus,SocialComfort,InfrastructureObjects,PowerPlants,WaterIntakes,TreatmentFacilities,Reservoirs,GasStationsDist,InvestmentReturn
0,residential,2154.0,23.680,6.08800,307.405291,4.004826,239.520000,0.11300,18.632,91.17300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.149818
1,recreation,2154.0,23.680,6.08800,307.405291,4.004826,239.520000,0.11300,18.632,91.17300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.082828
2,special,2154.0,23.680,6.08800,307.405291,4.004826,239.520000,0.11300,18.632,91.17300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.444405
3,industrial,2154.0,23.680,6.08800,307.405291,4.004826,239.520000,0.11300,18.632,91.17300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.698250
4,agriculture,2154.0,23.680,6.08800,307.405291,4.004826,239.520000,0.11300,18.632,91.17300,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.746904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2452,residential_individual,45697.0,47.015,12.65525,280.033159,4.384311,262.577625,0.24275,13.360,21.75225,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.870863
2453,residential_lowrise,45697.0,47.015,12.65525,280.033159,4.384311,262.577625,0.24275,13.360,21.75225,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.604405
2454,residential_midrise,45697.0,47.015,12.65525,280.033159,4.384311,262.577625,0.24275,13.360,21.75225,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.321699
2455,residential_multistorey,45697.0,47.015,12.65525,280.033159,4.384311,262.577625,0.24275,13.360,21.75225,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.677616


In [47]:

# Убираем экстремальные значения в целевой переменной
df['InvestmentReturn'] = df['InvestmentReturn'].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['InvestmentReturn'])
df['InvestmentReturn'] = df['InvestmentReturn'].clip(lower=-100, upper=100)  # Ограничиваем разумный диапазон

# Подготовка данных для обучения
df['LandUseType'] = le.transform(df['LandUseType'])
X = df.drop('InvestmentReturn', axis=1)
y = df['InvestmentReturn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучение модели
model = XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.2, alpha=1.0, random_state=42)
model.fit(X_train, y_train)

# Оценка модели
print(f"Точность на обучающей выборке (R²): {model.score(X_train, y_train):.3f}")
print(f"Точность на тестовой выборке (R²): {model.score(X_test, y_test):.3f}")

Точность на обучающей выборке (R²): 0.669
Точность на тестовой выборке (R²): 0.629


In [49]:
def predict_investment_potential(territory_idx):
    territory_data = gdf.iloc[territory_idx]
    predictions = {}
    for land_type in land_use_types:
        input_data = pd.DataFrame([{
            'LandUseType': le.transform([land_type])[0],
            'Population': features['Population'].iloc[territory_idx],
            'Urbanization': features['Urbanization'].iloc[territory_idx],
            'HighwayAccessibility': features['HighwayAccessibility'].iloc[territory_idx],
            'RegionCenterAccessibility': features['RegionCenterAccessibility'].iloc[territory_idx],
            'Connectivity': features['Connectivity'].iloc[territory_idx],
            'SettlementConnectivity': features['SettlementConnectivity'].iloc[territory_idx],
            'RoadDensity': features['RoadDensity'].iloc[territory_idx],
            'FederalRoads': features['FederalRoads'].iloc[territory_idx],
            'RegionalRoads': features['RegionalRoads'].iloc[territory_idx],
            'LocalRoads': features['LocalRoads'].iloc[territory_idx],
            'PublicTransportRoutes': features['PublicTransportRoutes'].iloc[territory_idx],
            'PublicTransportStops': features['PublicTransportStops'].iloc[territory_idx],
            'GasStations': features['GasStations'].iloc[territory_idx],
            'GasStationAccessibility': features['GasStationAccessibility'].iloc[territory_idx],
            'RailwayLength': features['RailwayLength'].iloc[territory_idx],
            'RailwayStops': features['RailwayStops'].iloc[territory_idx],
            'RailwayAccessibility': features['RailwayAccessibility'].iloc[territory_idx],
            'IntAirports': features['IntAirports'].iloc[territory_idx],
            'LocalAirports': features['LocalAirports'].iloc[territory_idx],
            'IntAirportAccessibility': features['IntAirportAccessibility'].iloc[territory_idx],
            'LocalAirportAccessibility': features['LocalAirportAccessibility'].iloc[territory_idx],
            'Ports': features['Ports'].iloc[territory_idx],
            'Kindergartens': features['Kindergartens'].iloc[territory_idx],
            'Schools': features['Schools'].iloc[territory_idx],
            'ChildrenCenters': features['ChildrenCenters'].iloc[territory_idx],
            'ArtSchools': features['ArtSchools'].iloc[territory_idx],
            'MedicalPoints': features['MedicalPoints'].iloc[territory_idx],
            'Pharmacies': features['Pharmacies'].iloc[territory_idx],
            'Clinics': features['Clinics'].iloc[territory_idx],
            'LocalHospitals': features['LocalHospitals'].iloc[territory_idx],
            'CityHospitals': features['CityHospitals'].iloc[territory_idx],
            'CommercialClinics': features['CommercialClinics'].iloc[territory_idx],
            'GroceryStores': features['GroceryStores'].iloc[territory_idx],
            'HardwareStores': features['HardwareStores'].iloc[territory_idx],
            'ShoppingCenters': features['ShoppingCenters'].iloc[territory_idx],
            'PetStores': features['PetStores'].iloc[territory_idx],
            'Hypermarkets': features['Hypermarkets'].iloc[territory_idx],
            'SpecialtyStores': features['SpecialtyStores'].iloc[territory_idx],
            'Cafes': features['Cafes'].iloc[territory_idx],
            'Restaurants': features['Restaurants'].iloc[territory_idx],
            'FoodCourts': features['FoodCourts'].iloc[territory_idx],
            'CommunityHalls': features['CommunityHalls'].iloc[territory_idx],
            'CultureCenters': features['CultureCenters'].iloc[territory_idx],
            'Libraries': features['Libraries'].iloc[territory_idx],
            'ReligiousSites': features['ReligiousSites'].iloc[territory_idx],
            'Playgrounds': features['Playgrounds'].iloc[territory_idx],
            'Parks': features['Parks'].iloc[territory_idx],
            'PublicSpaces': features['PublicSpaces'].iloc[territory_idx],
            'LargeParks': features['LargeParks'].iloc[territory_idx],
            'AmusementParks': features['AmusementParks'].iloc[territory_idx],
            'DogParks': features['DogParks'].iloc[territory_idx],
            'WorkoutAreas': features['WorkoutAreas'].iloc[territory_idx],
            'FitnessCenters': features['FitnessCenters'].iloc[territory_idx],
            'SkateParks': features['SkateParks'].iloc[territory_idx],
            'SportsComplexes': features['SportsComplexes'].iloc[territory_idx],
            'DeliveryPoints': features['DeliveryPoints'].iloc[territory_idx],
            'Salons': features['Salons'].iloc[territory_idx],
            'HouseholdServices': features['HouseholdServices'].iloc[territory_idx],
            'Banks': features['Banks'].iloc[territory_idx],
            'TransportStops': features['TransportStops'].iloc[territory_idx],
            'Parking': features['Parking'].iloc[territory_idx],
            'FuelStations': features['FuelStations'].iloc[territory_idx],
            'TrainStations': features['TrainStations'].iloc[territory_idx],
            'CarServices': features['CarServices'].iloc[territory_idx],
            'PolicePoints': features['PolicePoints'].iloc[territory_idx],
            'PoliceStations': features['PoliceStations'].iloc[territory_idx],
            'FireStations': features['FireStations'].iloc[territory_idx],
            'PublicTransportConnectivity': features['PublicTransportConnectivity'].iloc[territory_idx],
            'FederalCenterAccessibility': features['FederalCenterAccessibility'].iloc[territory_idx],
            'DistrictCenterAccessibility': features['DistrictCenterAccessibility'].iloc[territory_idx],
            'SocialBasic': features['SocialBasic'].iloc[territory_idx],
            'SocialPlus': features['SocialPlus'].iloc[territory_idx],
            'SocialComfort': features['SocialComfort'].iloc[territory_idx],
            'InfrastructureObjects': features['InfrastructureObjects'].iloc[territory_idx],
            'PowerPlants': features['PowerPlants'].iloc[territory_idx],
            'WaterIntakes': features['WaterIntakes'].iloc[territory_idx],
            'TreatmentFacilities': features['TreatmentFacilities'].iloc[territory_idx],
            'Reservoirs': features['Reservoirs'].iloc[territory_idx],
            'GasStationsDist': features['GasStationsDist'].iloc[territory_idx]
        }])
        
        # Обрабатываем NaN в input_data
        for col in input_data.columns:
            input_data[col] = pd.to_numeric(input_data[col], errors='coerce')
            input_data[col] = input_data[col].fillna(df[col].median())
        
        # Предсказание с помощью модели
        pred = model.predict(input_data)[0]
        predictions[land_type] = pred
    
    return predictions

# Пример использования
territory_idx = 3
predictions = predict_investment_potential(territory_idx)
print(f"Инвестиционная привлекательность для территории {gdf.loc[territory_idx, 'name']}:")
for land_type, score in predictions.items():
    print(f"{land_type}: {score:.2f}")

Инвестиционная привлекательность для территории Борское сельское поселение:
residential: 0.93
recreation: 0.61
special: 0.65
industrial: 1.58
agriculture: 0.69
transport: 1.02
business: 1.86
basic: 0.26
residential_individual: 1.25
residential_lowrise: 1.29
residential_midrise: 1.32
residential_multistorey: 1.39
mixed_use: 1.77


In [50]:
# После обучения модели добавляем расчет и вывод топ территорий
# Предыдущий код остается без изменений до этого момента

# Функция предсказания для всех территорий с указанием лучшего типа землепользования
def get_top_territories(gdf, model, le, features, top_n=5):
    territory_scores = []
    
    for idx in range(len(gdf)):
        territory_name = gdf.loc[idx, 'name']
        predictions = predict_investment_potential(idx)  # Используем ранее определенную функцию
        # Находим максимальную привлекательность и соответствующий тип землепользования
        max_score = max(predictions.values())
        best_land_use = max(predictions, key=predictions.get)  # Тип с максимальным значением
        territory_scores.append({
            'territory_id': gdf.loc[idx, 'territory_id'],
            'name': territory_name,
            'max_investment_score': max_score,
            'best_land_use': best_land_use
        })
    
    # Преобразуем в DataFrame и сортируем
    scores_df = pd.DataFrame(territory_scores)
    top_territories = scores_df.sort_values(by='max_investment_score', ascending=False).head(top_n)
    
    return top_territories

# Получаем и выводим топ-5 территорий
top_5 = get_top_territories(gdf, model, le, features, top_n=5)
print("Топ-5 территорий по инвестиционной привлекательности:")
print(top_5)

# Опционально: вывод подробных предсказаний для лучшей территории
best_territory_idx = top_5.index[0]
print(f"\nПодробные предсказания для лучшей территории ({top_5.loc[best_territory_idx, 'name']}):")
best_predictions = predict_investment_potential(best_territory_idx)
for land_type, score in best_predictions.items():
    print(f"{land_type}: {score:.2f}")

Топ-5 территорий по инвестиционной привлекательности:
    territory_id                              name  max_investment_score  \
65            73    Гатчинское городское поселение              4.392319   
33            39     Муринское городское поселение              4.380739   
29            35     Заневское городское поселение              3.912670   
30            36  Всеволожское городское поселение              3.887107   
53            60    Выборгское городское поселение              3.693727   

   best_land_use  
65      business  
33      business  
29      business  
30      business  
53      business  

Подробные предсказания для лучшей территории (Гатчинское городское поселение):
residential: 3.99
recreation: 3.82
special: 3.51
industrial: 4.20
agriculture: 3.64
transport: 3.40
business: 4.39
basic: 3.54
residential_individual: 4.09
residential_lowrise: 4.09
residential_midrise: 4.22
residential_multistorey: 4.21
mixed_use: 4.37


# ML

In [42]:
# 1. Создание и обучение модели
# Инициализация LabelEncoder с известными значениями LandUseType
le = LabelEncoder()
le.fit(['residential', 'recreation', 'special', 'industrial', 
        'agriculture', 'transport', 'business', 'basic', 
        'residential_individual', 'residential_lowrise', 
        'residential_midrise', 'residential_multistorey',
        'mixed_use'])  # Обучаем на строковых метках

# Генерация данных для обучения модели
np.random.seed(42)
n_samples = 10000
data = {
    'LandUseType': np.random.choice(['residential', 'recreation', 'special', 'industrial', 
                                     'agriculture', 'transport', 'business', 'basic', 
                                     'residential_individual', 'residential_lowrise', 'residential_midrise', 
                                     'residential_multistorey', 'mixed_use'], n_samples),

    'LocationScore': np.random.uniform(1, 10, n_samples),
    'AreaTotal': np.random.uniform(100, 20000, n_samples),
    'AreaResidential': np.random.uniform(0, 15000, n_samples),
    'AreaCommercial': np.random.uniform(0, 10000, n_samples),
    'Infrastructure': np.random.uniform(1, 10, n_samples),
    'RoadDevelopment': np.random.uniform(1, 10, n_samples),
    'EcoRestrictions': np.random.uniform(0, 5, n_samples),
    'PopulationDensity': np.random.uniform(50, 5000, n_samples),
    'AvgIncomeLevel': np.random.uniform(20000, 100000, n_samples),
    'DistanceToCenter': np.random.uniform(0.5, 20, n_samples),
    'MarketDemand': np.random.uniform(1, 10, n_samples),
    'Taxes': np.random.uniform(1000, 5000, n_samples),
    'Income': np.random.uniform(5000, 20000, n_samples),
    'ProvisionLevel': np.random.uniform(1, 10, n_samples)
}

land_use_effect = {
    'residential': 0.7,  # Жилая зона – средняя привлекательность, стабильный спрос
    'recreation': 0.4,  # Рекреационная зона – низкая доходность, но ценно для экологии и туризма
    'special': 0.3,  # Особого назначения – ограниченное использование, низкая инвестиционная привлекательность
    'industrial': 1.2,  # Промышленная зона – стабильный доход, но зависимость от экономики
    'agriculture': 0.5,  # Сельскохозяйственная зона – низкая доходность, но долгосрочные инвестиции
    'transport': 0.8,  # Транспортная зона – средняя привлекательность, но зависимость от инфраструктуры
    'business': 2.0,  # Общественно-деловая зона – высокая инвестиционная привлекательность
    'basic': 0.2,  # Базовая зона – малоперспективная для инвестиций
    'residential_individual': 0.9,  # ИЖС – средняя доходность, зависит от спроса
    'residential_lowrise': 1.0,  # Малоэтажная жилая зона – хороший баланс доходности и стабильности
    'residential_midrise': 1.2,  # Среднеэтажная жилая зона – выше привлекательность, плотность застройки
    'residential_multistorey': 1.5,  # Многоэтажная жилая зона – высокая инвестиционная привлекательность
    'mixed_use': 1.8  # Многофункциональная зона – высокая привлекательность за счет гибкости использования
}


# Формула доходности с учетом новых признаков
data['InvestmentReturn'] = (
    0.4 * data['LocationScore'] +
    0.3 * data['Infrastructure'] +
    0.2 * data['RoadDevelopment'] -
    0.2 * data['EcoRestrictions'] +
    0.1 * np.log(data['PopulationDensity']) +
    0.2 * data['AvgIncomeLevel'] / 10000 -
    0.3 * data['DistanceToCenter'] +
    0.3 * data['MarketDemand'] +
    0.3 * (data['AreaTotal'] / 10000) +
    0.2 * (data['AreaResidential'] / 10000) +
    0.4 * (data['AreaCommercial'] / 10000) -
    0.3 * (data['Taxes'] / 1000) +
    0.4 * (data['Income'] / 10000) +
    0.2 * data['ProvisionLevel'] +
    [land_use_effect[land] for land in data['LandUseType']] +
    np.random.normal(0, 0.5, n_samples)
)
df = pd.DataFrame(data)

# Подготовка данных для обучения
df['LandUseType'] = le.transform(df['LandUseType'])  # Кодируем строковые метки в числовые
X = df.drop('InvestmentReturn', axis=1)
y = df['InvestmentReturn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Обучение модели
model = XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, alpha=1.0, random_state=42)
model.fit(X_train, y_train)

# Оценка
print(f"Точность на обучающей выборке (R²): {model.score(X_train, y_train):.3f}")
print(f"Точность на тестовой выборке (R²): {model.score(X_test, y_test):.3f}")

Точность на обучающей выборке (R²): 0.862
Точность на тестовой выборке (R²): 0.837


In [47]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import shap
from shapely.geometry import Polygon

# Подготовка имеющегося GeoDataFrame
territory_gdf = split_gdf.copy()  # Копируем, чтобы избежать изменений в оригинале

# Добавляем необходимые атрибуты для модели (случайные данные для примера)
np.random.seed(42)  # Фиксируем генератор случайных чисел для воспроизводимости

territory_gdf['LandUseType'] = np.random.choice(['residential', 'recreation', 'special', 'industrial', 
        'agriculture', 'transport', 'business', 'basic', 
        'residential_individual', 'residential_lowrise', 
        'residential_midrise', 'residential_multistorey',
        'mixed_use'], len(territory_gdf))

territory_gdf['LocationScore'] = np.random.uniform(1, 10, len(territory_gdf))
territory_gdf['AreaTotal'] = np.random.uniform(100, 20000, len(territory_gdf))
territory_gdf['AreaResidential'] = np.random.uniform(0, 15000, len(territory_gdf))
territory_gdf['AreaCommercial'] = np.random.uniform(0, 10000, len(territory_gdf))
territory_gdf['Infrastructure'] = np.random.uniform(1, 10, len(territory_gdf))
territory_gdf['RoadDevelopment'] = np.random.uniform(1, 10, len(territory_gdf))
territory_gdf['EcoRestrictions'] = np.random.uniform(0, 5, len(territory_gdf))
territory_gdf['PopulationDensity'] = np.random.uniform(50, 5000, len(territory_gdf))
territory_gdf['AvgIncomeLevel'] = np.random.uniform(20000, 100000, len(territory_gdf))
territory_gdf['DistanceToCenter'] = np.random.uniform(0.5, 20, len(territory_gdf))
territory_gdf['MarketDemand'] = np.random.uniform(1, 10, len(territory_gdf))
territory_gdf['Taxes'] = np.random.uniform(1000, 5000, len(territory_gdf))
territory_gdf['Income'] = np.random.uniform(5000, 20000, len(territory_gdf))
territory_gdf['ProvisionLevel'] = np.random.uniform(1, 10, len(territory_gdf))

# Кодирование LandUseType для модели
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
territory_gdf['LandUseType'] = le.fit_transform(territory_gdf['LandUseType'])

feature_columns = ['LandUseType', 'LocationScore', 'AreaTotal', 'AreaResidential', 
                   'AreaCommercial', 'Infrastructure', 'RoadDevelopment', 
                   'EcoRestrictions', 'PopulationDensity', 'AvgIncomeLevel', 
                   'DistanceToCenter', 'MarketDemand', 'Taxes', 'Income', 'ProvisionLevel']

# Убираем ненужные столбцы
X_new = territory_gdf[feature_columns]
investment_returns = model.predict(X_new)

# Добавляем предсказания в GeoDataFrame
territory_gdf['InvestmentReturn'] = investment_returns


# Добавляем предсказания в GeoDataFrame
territory_gdf['InvestmentReturn'] = investment_returns

# Декодируем LandUseType обратно для визуализации
territory_gdf['LandUseType'] = le.inverse_transform(territory_gdf['LandUseType'])

# 🔹 Визуализация предсказаний на карте
fig, ax = plt.subplots(figsize=(30, 20))
territory_gdf.plot(column='InvestmentReturn', ax=ax, cmap='RdYlGn', legend=True, 
                    edgecolor='black', alpha=0.7, legend_kwds={'label': "Доходность (%)"})

plt.title("Карта территорий с предсказанной доходностью и типом землепользования")

# Аннотации
for idx, row in territory_gdf.iterrows():
    centroid = row['geometry'].centroid
    plt.annotate(f"{row['InvestmentReturn']:.2f}%\n{row['LandUseType']}", 
                 xy=centroid.coords[0], 
                 ha='center', va='center', fontsize=8, 
                 bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))

plt.axis('off')
plt.show()

# 🔹 SHAP-анализ для объяснения модели
explainer = shap.TreeExplainer(model)

# Анализ SHAP для первой территории
new_territory = X_new.iloc[[0]]  
shap_values_new = explainer.shap_values(new_territory.values)

# Бар-график SHAP
plt.figure(figsize=(12, 6))
plt.bar(new_territory.columns, shap_values_new[0], 
        color=['green' if x > 0 else 'red' for x in shap_values_new[0]])
plt.axhline(0, color='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.xlabel("Признаки")
plt.ylabel("Вклад в доходность (%)")
plt.title(f"Влияние признаков на доходность для территории 0 ({territory_gdf['LandUseType'].iloc[0]})")
plt.tight_layout()
plt.show()

shap.waterfall_plot(shap.Explanation(values=shap_values_new[0], 
                                     base_values=explainer.expected_value, 
                                     data=new_territory.iloc[0], 
                                     feature_names=new_territory.columns.tolist()))

# Вывод SHAP-вкладов
print("\nВклад признаков в прогноз для новой территории:")
for feature, shap_val in zip(new_territory.columns, shap_values_new[0]):
    print(f"{feature}: {shap_val:.3f}")


NameError: name 'split_gdf' is not defined

In [8]:
# mo_territories_gdf = gpd.read_parquet('data/gnn_data/territories.parquet')
# mo_territories_gdf

In [9]:
# block = gpd.read_parquet('data/gnn_data/189_blocks.parquet')
# block

In [10]:
# import geopandas as gpd
# import pandas as pd
# import os

# # Путь к файлам Parquet
# parquet_dir = "data/gnn_data"

# # Целевая CRS (можно заменить на нужную)
# target_crs = "EPSG:32636"

# # Создаем пустой список для хранения загруженных геоданных
# all_blocks = []

# # Проходим по всем territory_id
# for territory_id in mo_territories_gdf.index:
#     parquet_path = os.path.join(parquet_dir, f"{territory_id}_blocks.parquet")
    
#     # Проверяем, существует ли файл
#     if os.path.exists(parquet_path):
#         try:
#             block_gdf = gpd.read_parquet(parquet_path)
            
#             # Проверяем, есть ли у GeoDataFrame CRS, и приводим к целевой системе координат
#             if block_gdf.crs and block_gdf.crs != target_crs:
#                 block_gdf = block_gdf.to_crs(target_crs)

#             all_blocks.append(block_gdf)
#         except Exception as e:
#             print(f"Ошибка при загрузке {parquet_path}: {e}")
#     else:
#         print(f"Файл не найден: {parquet_path}")

# # Объединяем все загруженные файлы в один GeoDataFrame
# if all_blocks:
#     merged_gdf = gpd.GeoDataFrame(pd.concat(all_blocks, ignore_index=True), crs=target_crs)
#     print("Объединенный GeoDataFrame создан успешно!")
# else:
#     merged_gdf = gpd.GeoDataFrame()
#     print("Не удалось загрузить ни одного файла.")

# merged_gdf